# Tome 3, Chapter 11: Brownian Motion (Python companion)

Exercises first, solutions at the end. We build Brownian paths in two ways, measure their quadratic variation, find the hidden bias in the simulated maximum, watch binomial option prices converge to Black–Scholes, price options by Monte Carlo with variance reduction, and hedge an option by trading the stock: the hedging error shrinks as we trade more often, whatever the real drift of the stock.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(11)
Phi = np.vectorize(lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2))))

## Exercises

**P11.1 (★) Two constructions.** Build $10^4$ Brownian paths on $[0,1]$ with $2^{10}$ steps in two ways: by cumulative sums of $N(0,2^{-10})$ increments, and by Lévy's midpoint displacement (Theorem 11.4). Check that both give $\operatorname{Var}B_t\approx t$ and $\operatorname{Cov}(B_{0.3},B_{0.7})\approx0.3$.

**P11.2 (★) Variations.** On one path with $2^{20}$ steps, compute the dyadic sums $\sum|\Delta B|^p$ for $p=1,2,3$ at levels $n=4,\dots,20$ (Theorem 11.8, Exercise 11.5).

**P11.3 (★★) The bias of a simulated maximum.** By Theorem 11.12, $\mathbb E[M_1]=\mathbb E|B_1|=\sqrt{2/\pi}$. Estimate the mean of the maximum of a path observed only at $n$ equally spaced times, by simulation for $n=2^2,\dots,2^8$, and exactly up to $n=2^{20}$ with Spitzer's identity $\mathbb E[\max_{k\le n}S_k]=\sum_{k\le n}\frac{\mathbb E[S_k^+]}k$ (with $S_0=0$). How does the bias decrease with $n$?

**P11.4 (★) Binomial prices.** For $S_0=K=100$, $r=0.05$, $\sigma=0.2$, $T=1$, compute the Cox–Ross–Rubinstein call price for $n=1,\dots,300$ periods and plot its error against Black–Scholes (Theorem 11.17).

**P11.5 (★★) Monte Carlo pricing.** Price the same call by simulating $S_T=S_0e^{(r-\sigma^2/2)T+\sigma\sqrt TZ}$, with a $95\%$ confidence interval, for $10^4$, $10^5$ and $10^6$ samples. Reduce the variance with antithetic variables ($Z$ and $-Z$) and with the control variate $S_T$, whose mean $S_0e^{rT}$ is known. Then price an *Asian* call, paying $(\bar S-K)^+$ where $\bar S$ is the average price over $252$ trading days, for which there is no closed formula.

**P11.6 (★★) Hedging in practice.** A bank sells the call at the Black–Scholes price and hedges by holding $\Phi(d_+)$ shares (with $d_+$ computed from the current price and remaining time), rebalanced $n$ times over the year, the rest in the bank. Simulate the final profit and loss for $n=12,52,252,1000$ when the stock really grows at the rate $\mu=0.15$ (not $r$). What happens to its mean and spread?

## Solutions

**P11.1.**

In [ ]:
def bm_cumsum(paths, n):
    return np.hstack([np.zeros((paths, 1)), np.cumsum(rng.standard_normal((paths, 2**n)) * 2**(-n / 2), axis=1)])
def bm_levy(paths, n):
    B = np.zeros((paths, 2)); B[:, 1] = rng.standard_normal(paths)
    for m in range(1, n + 1):
        new = np.empty((paths, 2 * B.shape[1] - 1))
        new[:, 0::2] = B
        new[:, 1::2] = (B[:, :-1] + B[:, 1:]) / 2 + 2**(-(m + 1) / 2) * rng.standard_normal((paths, B.shape[1] - 1))
        B = new
    return B
n = 10; i3, i7 = round(0.3 * 2**n), round(0.7 * 2**n)
for name, f in [('cumulative sums', bm_cumsum), ('Levy midpoints', bm_levy)]:
    B = f(10_000, n)
    print(f'{name:16s} Var B_0.5 = {B[:, 2**(n-1)].var():.4f}   Var B_1 = {B[:, -1].var():.4f}   Cov(B_0.3, B_0.7) = {np.cov(B[:, i3], B[:, i7])[0, 1]:.4f}')
t = np.linspace(0, 1, 2**n + 1)
plt.figure(figsize=(7, 3)); plt.plot(t, bm_levy(5, n).T, lw=0.7); plt.xlabel('t'); plt.tight_layout(); plt.show()

Both constructions produce the same law on the grid, as they must (Lemma 11.3); the covariance is $\min(s,t)$ up to sampling error ($0.3$ is the grid time $307/1024$). The midpoint construction has a practical advantage: its first levels already capture the large-scale shape of the path, so it can be refined on demand. In quasi-Monte Carlo methods for finance, this "Brownian bridge ordering" puts the most important random numbers first and greatly improves convergence.

**P11.2.**

In [ ]:
N = 20
B = np.concatenate([[0], np.cumsum(rng.standard_normal(2**N) * 2**(-N / 2))])
print(' n    sum|dB|      sum|dB|^2    sum|dB|^3')
for n in range(4, N + 1, 2):
    d = np.diff(B[::2**(N - n)])
    print(f'{n:2d}  {np.abs(d).sum():10.3f}  {np.sum(d**2):10.4f}  {np.sum(np.abs(d)**3):10.5f}')

The quadratic variation settles at $1$, while the length of the polygonal path grows like $2^{n/2}\sqrt{2/\pi}$ (about $820$ at level $20$) and the cubic sums vanish like $2^{-n/2}$. A Brownian path seen at finer and finer scales becomes infinitely long, yet its squared increments add up to exactly the elapsed time. This is the fact that makes the volatility of a stock measurable from high-frequency data: the realised variance $\sum(\Delta\ln S)^2$ estimates $\sigma^2T$ better and better as the sampling becomes finer.

**P11.3.**

In [ ]:
print('exact E[M_1] =', math.sqrt(2 / math.pi))
print('    n   simulated mean max   Spitzer exact    bias    bias*sqrt(n)')
for k in range(2, 21, 2):
    n = 2**k
    exact_grid = sum(k**-0.5 for k in range(1, n + 1)) / math.sqrt(2 * math.pi * n)
    sim = ''
    if n <= 256:
        reps = 40000; M = np.zeros(reps); S = np.zeros(reps)
        for j in range(n):
            S += rng.standard_normal(reps) / math.sqrt(n)
            np.maximum(M, S, out=M)
        sim = f'{M.mean():.4f}'
    bias = math.sqrt(2 / math.pi) - exact_grid
    print(f'{n:8d}   {sim:>8s}            {exact_grid:.5f}       {bias:.5f}   {bias * math.sqrt(n):.4f}')
print('-zeta(1/2)/sqrt(2 pi) =', 1.4603545088095868 / math.sqrt(2 * math.pi))

The grid misses the peaks between observation times. Simulation is too noisy to measure a small bias, but the mean of the maximum of a random walk has an exact formula, *Spitzer's identity* $\mathbb E[\max_{k\le n}S_k]=\sum_{k\le n}\frac{\mathbb E[S_k^+]}k$, which here gives $\frac1{\sqrt{2\pi n}}\sum_{k\le n}k^{-1/2}$. The bias decreases only like $\frac c{\sqrt n}$, with $c=-\zeta(\frac12)/\sqrt{2\pi}\approx0.5826$, since $\sum_{k\le n}k^{-1/2}=2\sqrt n+\zeta(\frac12)+O(n^{-1/2})$. Asmussen, Glynn and Pitman (1995) and Broadie, Glasserman and Kou (1997) used this constant to correct the price of barrier options monitored at discrete times: shift the barrier by $0.5826\,\sigma\sqrt{\Delta t}$. Without the correction, halving the bias costs four times as many time steps.

**P11.4.**

In [ ]:
from math import comb
S0, K, r, sig, T = 100.0, 100.0, 0.05, 0.2, 1.0
def bs_call(S0, K, r, sig, T):
    dp = (math.log(S0 / K) + (r + sig**2 / 2) * T) / (sig * math.sqrt(T)); dm = dp - sig * math.sqrt(T)
    return S0 * Phi(dp) - K * math.exp(-r * T) * Phi(dm)
def crr_call(n):
    h = T / n; u = math.exp(sig * math.sqrt(h)); d = 1 / u; p = (math.exp(r * h) - d) / (u - d)
    k = np.arange(n + 1)
    ST = S0 * u**k * d**(n - k)
    logw = np.array([math.lgamma(n + 1) - math.lgamma(j + 1) - math.lgamma(n - j + 1) for j in k]) + k * math.log(p) + (n - k) * math.log(1 - p)
    return math.exp(-r * T) * np.sum(np.exp(logw) * np.maximum(ST - K, 0))
C = bs_call(S0, K, r, sig, T)
ns = np.arange(1, 301)
err = np.array([crr_call(n) - C for n in ns])
print('Black-Scholes:', C, '   CRR n=1000:', crr_call(1000))
plt.figure(figsize=(7, 3)); plt.plot(ns, err, lw=0.8); plt.plot(ns, 1 / ns, 'k--', lw=0.8); plt.plot(ns, -1 / ns, 'k--', lw=0.8)
plt.ylim(-0.25, 0.25); plt.xlabel('periods n'); plt.ylabel('CRR - BS'); plt.tight_layout(); plt.show()

The error oscillates between even and odd $n$ and decays like $\frac1n$ (the dashed lines are $\pm\frac1n$). The oscillation comes from the position of the strike between two terminal prices of the tree, which changes with $n$: the payoff has a kink at $K$. Averaging two consecutive $n$, or placing the strike on a node, removes most of it; for American options, where no closed formula exists, the tree remains a standard tool.

**P11.5.**

In [ ]:
def mc_call(N, method='plain'):
    Z = rng.standard_normal(N)
    drift, vol = (r - sig**2 / 2) * T, sig * math.sqrt(T)
    disc = math.exp(-r * T)
    if method == 'antithetic':
        Y = disc * 0.5 * (np.maximum(S0 * np.exp(drift + vol * Z) - K, 0) + np.maximum(S0 * np.exp(drift - vol * Z) - K, 0))
    else:
        ST = S0 * np.exp(drift + vol * Z)
        Y = disc * np.maximum(ST - K, 0)
        if method == 'control':
            Xc = ST - S0 * math.exp(r * T)
            b = np.cov(Y, Xc)[0, 1] / Xc.var()
            Y = Y - b * Xc
    return Y.mean(), 1.96 * Y.std() / math.sqrt(N)
print('Black-Scholes', round(C, 4))
for N in [10**4, 10**5, 10**6]:
    line = f'N={N:>8d}'
    for m in ['plain', 'antithetic', 'control']:
        est, hw = mc_call(N, m)
        line += f'   {m}: {est:.3f} +- {hw:.3f}'
    print(line)

In [ ]:
days, N = 252, 100_000
dt = T / days
logS = np.log(S0) + np.cumsum((r - sig**2 / 2) * dt + sig * math.sqrt(dt) * rng.standard_normal((N, days)), axis=1)
avg = np.exp(logS).mean(axis=1)
Y = math.exp(-r * T) * np.maximum(avg - K, 0)
print('Asian call:', Y.mean(), '+-', 1.96 * Y.std() / math.sqrt(N), '   (European call:', round(C, 3), ')')

Monte Carlo converges like $\frac1{\sqrt N}$: each extra digit costs $100$ times more samples, as Chapter 6 predicted. Antithetic variables halve the half-width (but each sample costs two payoffs, so the real gain is a factor $2$ in cost), and the control variate divides it by about $2.6$, for the price of one covariance: equivalent to $7$ times more samples. One of the nine intervals misses the exact value, as about one in twenty should. The Asian call is worth about $5.8$, much less than the European $10.45$, because an average fluctuates less than a final price: options on less volatile quantities are cheaper. No formula gives this number, and Monte Carlo is how banks compute it.

**P11.6.**

In [ ]:
def hedge(n, mu, N=20_000):
    dt = T / n
    S = np.full(N, S0)
    def delta(S, tau):
        dp = (np.log(S / K) + (r + sig**2 / 2) * tau) / (sig * np.sqrt(tau))
        return Phi(dp)
    shares = delta(S, T)
    cash = C - shares * S                     # sell the call at C, buy the shares
    for k in range(1, n + 1):
        S = S * np.exp((mu - sig**2 / 2) * dt + sig * math.sqrt(dt) * rng.standard_normal(N))
        cash *= math.exp(r * dt)
        if k < n:
            new = delta(S, T - k * dt)
            cash -= (new - shares) * S
            shares = new
    return cash + shares * S - np.maximum(S - K, 0)
print('   n    mean P&L    std P&L    std*sqrt(n)')
for n in [12, 52, 252, 1000]:
    pl = hedge(n, 0.15)
    print(f'{n:5d}   {pl.mean():+.4f}    {pl.std():.4f}     {pl.std() * math.sqrt(n):.2f}')
pl = hedge(252, -0.10)
print('with mu = -0.10, n = 252: mean', round(pl.mean(), 4), ' std', round(pl.std(), 4))

The profit and loss has mean close to $0$ and a spread that decreases like $\frac1{\sqrt n}$: with daily rebalancing it is about $0.4$ on a call worth $10.45$, and it would vanish in the continuous limit, where the hedge replicates the option exactly. The real growth rate $\mu$ makes no difference, as in the binomial model (Theorem 11.16): the hedge removes the exposure to the direction of the stock, and only its volatility matters. In practice, transaction costs forbid very frequent rebalancing, and the residual risk is what traders are paid to manage.